# Experiment C2 — Double Inverted Pendulum Control 🎪
**AI Builders Lab | Google Colab | Reinforcement Learning**

A cart supports **two connected pendulum links**. Can an AI learn to keep both upright?

This is a **more challenging** control task than CartPole. We use the Gymnasium **InvertedDoublePendulum-v5** MuJoCo simulator and PPO.

**Run:** Runtime → Run all. **CPU is supported.** This project may require **several to tens of minutes**, depending on Colab hardware and training steps. Successful balancing is **not guaranteed** at the default training budget.

> The pendulum starts near the upright position. This is primarily a **balancing/stabilization** experiment, not a swing-up-from-hanging-down experiment. PPO learns a policy; it does not learn an explicit dynamics world model.

## 1. What changes from CartPole?
- CartPole: one pole, **two discrete actions** (left/right).
- Double inverted pendulum: two linked poles, **one continuous horizontal force**.
- The controller must account for coupled motion: moving the cart affects **both links**.

**RL loop:** state → policy → continuous cart force → MuJoCo physics → reward → updated state.

**Physics note:** The two links are mechanically coupled. Stabilizing one link alone is not enough.

In [ ]:
%pip -q install "stable-baselines3>=2.3,<3" "gymnasium[mujoco]>=1.0,<2" matplotlib

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.utils import set_random_seed

SEED = 42
ENV_ID = "InvertedDoublePendulum-v5"
TRAIN_STEPS = 60_000  # Start with 50_000 to test; try 600_000+ for stronger learning
VIDEO_FILE = "double_pendulum_demo.mp4"
VIDEO_MAX_STEPS = 500
set_random_seed(SEED)

try:
    env = gym.make(ENV_ID)
except Exception as exc:
    raise RuntimeError("MuJoCo environment failed to load. Restart Colab runtime after pip install and rerun.") from exc

observation, info = env.reset(seed=SEED)
print("Observation shape:", observation.shape)
print("Observation example:", observation.round(3))
print("Action space:", env.action_space)
print("Action range:", env.action_space.low, env.action_space.high)
env.close()

## 2. Random-force baseline
Measure how long a cart with random forces survives. Do not expect random control to balance the system.

In [ ]:
baseline_env = gym.make(ENV_ID)
random_scores = []
for episode in range(10):
    obs, _ = baseline_env.reset(seed=SEED + episode)
    total = 0.
    for step in range(1000):
        action = baseline_env.action_space.sample()
        obs, reward, terminated, truncated, _ = baseline_env.step(action)
        total += reward
        if terminated or truncated:
            break
    random_scores.append(total)
baseline_env.close()
print(f"Random baseline: {np.mean(random_scores):.1f} ± {np.std(random_scores):.1f}")

## 3. Train a continuous-control policy with PPO
PPO outputs a **continuous action** (cart force), not a left/right class. MuJoCo computes the next physical state.

**Experiment:** Compare the training curve and evaluation with the simpler CartPole notebook.

The training budget is a starting point, not a guarantee of convergence.

In [ ]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
import matplotlib.pyplot as plt
import numpy as np

class EpisodeLogger(BaseCallback):
    """Collect completed episode rewards and lengths from Monitor."""
    def __init__(self):
        super().__init__()
        self.rewards = []
        self.lengths = []
    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.rewards.append(float(info["episode"]["r"]))
                self.lengths.append(int(info["episode"]["l"]))
        return True

def plot_learning(log):
    if not log.rewards:
        print("No completed episodes recorded yet.")
        return
    window = min(25, len(log.rewards))
    smoothed = np.convolve(log.rewards, np.ones(window)/window, mode="valid")
    plt.figure(figsize=(9, 4))
    plt.plot(log.rewards, alpha=.25, label="Episode reward")
    plt.plot(np.arange(window-1,len(log.rewards)),smoothed,label=f"{window}-episode average")
    plt.xlabel("Completed training episodes")
    plt.ylabel("Total episode reward")
    plt.title("Reinforcement learning progress")
    plt.legend()
    plt.grid(alpha=.25)
    plt.show()

In [ ]:
train_env = Monitor(gym.make(ENV_ID))
train_env.reset(seed=SEED)
log = EpisodeLogger()
agent = PPO(
    "MlpPolicy", train_env,
    policy_kwargs=dict(net_arch=[64, 64]),
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=256,
    n_epochs=5,
    gamma=0.99,
    seed=SEED,
    verbose=0,
    device="cpu",
)
agent.learn(total_timesteps=TRAIN_STEPS, callback=log)
agent.save("double_pendulum_ppo_policy")
print("Saved trained policy immediately after training.")
train_env.close()
plot_learning(log)

## 4. Evaluate on fresh episodes
Rewards are **not directly comparable** between CartPole and the double pendulum because the reward functions differ. Compare each trained policy against its own random baseline instead.

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy
eval_env = Monitor(gym.make(ENV_ID))
mean_reward, std_reward = evaluate_policy(agent, eval_env, n_eval_episodes=10, deterministic=True)
print(f"Mean evaluation reward: {mean_reward:.1f} ± {std_reward:.1f}")
eval_env.close()

## 5. Watch the double pendulum — no OpenGL required

This animation draws the cart and both links from MuJoCo's **physics coordinates** using Matplotlib, without calling `env.render()` or creating any graphics context. The trained policy is saved immediately after training and automatically reloaded if the `agent` variable is missing. A runtime restart clears local files unless you saved the model externally.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
from matplotlib import pyplot as plt, animation
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO

MODEL_FILE = Path("double_pendulum_ppo_policy.zip")
if "agent" not in globals():
    if not MODEL_FILE.exists():
        raise FileNotFoundError(
            "No trained agent in memory or saved model. Run the training cell first. "
            "If the Colab runtime restarted, local files may have been erased."
        )
    agent = PPO.load(str(MODEL_FILE), device="cpu")
    print("Loaded saved PPO policy.")

# Physics simulation only: do NOT use render_mode or env.render().
sim = gym.make(ENV_ID)
obs, _ = sim.reset(seed=SEED + 1)
positions = []
forces = []
for step in range(VIDEO_MAX_STEPS):
    # MuJoCo's actual body positions: cart, first pole tip, second pole tip.
    data = sim.unwrapped.data
    positions.append(np.array([
        data.body("cart").xpos[[0, 2]],
        data.body("pole").xpos[[0, 2]],
        data.body("pole2").xpos[[0, 2]],
    ]))
    action, _ = agent.predict(obs, deterministic=True)
    forces.append(float(np.asarray(action).ravel()[0]))
    obs, reward, terminated, truncated, _ = sim.step(action)
    if terminated or truncated:
        break
sim.close()
positions = np.asarray(positions)
print(f"Recorded {len(positions)} physics steps without OpenGL.")

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-0.3, 2.5)
ax.set_aspect("equal")
ax.grid(alpha=0.2)
ax.axhline(0, color="gray", lw=1)
cart_dot, = ax.plot([], [], "ks", markersize=12)
links, = ax.plot([], [], "o-", linewidth=3, markersize=6)
force_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
def update(i):
    xy = positions[i]
    cart_dot.set_data([xy[0,0]], [xy[0,1]])
    links.set_data(xy[:,0], xy[:,1])
    force_text.set_text(f"Step {i+1}/{len(positions)} | Force {forces[i]:+.2f}")
    return cart_dot, links, force_text

# Downsample for compact, quick HTML animation.
frame_ids = np.linspace(0, len(positions)-1, min(len(positions), 160), dtype=int)
movie = animation.FuncAnimation(fig, update, frames=frame_ids, interval=50, blit=True)
plt.close(fig)
display(HTML(movie.to_jshtml()))

## 6. Saved policy

The training cell already saves `double_pendulum_ppo_policy.zip` immediately after training. You may save it to Google Drive or download it if you want to reuse it after a Colab runtime restart.

In [ ]:
from pathlib import Path
print("Model saved:", Path("double_pendulum_ppo_policy.zip").exists())

## 7. Discussion and extensions
1. Why is controlling **two linked poles** harder than one?
2. What is the difference between **discrete** and **continuous** action spaces?
3. Why does the agent need velocity information in addition to position?
4. Why shouldn't you compare CartPole's reward number directly with this environment's reward number?
5. Try `TRAIN_STEPS = 50_000`, then `300_000`. What changes?
6. Try `net_arch=[64, 64]`. Is the policy less capable?
7. **World Model connection:** MuJoCo is the simulator here. What would change if a neural network learned the next-state dynamics from transitions, and the controller planned using that learned model?

**Important distinction:** The simulator is a physics-based model of the environment, but this PPO agent does **not** learn its own explicit world model. A next project can add learned dynamics and model predictive control.

**Troubleshooting:** If the MuJoCo package fails to import immediately after installation, use **Runtime → Restart session**, then **Run all**. If the video encoding fails, rerun the video cell after the installation cell. Training performance varies by random seed and compute budget.